# Notebook 05 — Run the detector on your own data

If you have real EIS measurements from an MOE or MRE electrode, this notebook runs the full detection pipeline on your data.

## Where to put your data

There is a folder called `my_data/` in the same directory as this notebook. Place your data files there before running anything.

Two input formats are supported:

**Format A — Full EIS spectra (one CSV per measurement)**
Standard output from impedance analyzers (Gamry, BioLogic, Solartron, etc.).
Each file = one impedance sweep at one point in time. Name the files so they sort in chronological order (e.g. `measurement_001.csv`, `measurement_002.csv`, ...).

Each CSV file must have these three columns:

```
frequency_hz, re_z_ohm, neg_im_z_ohm
10000, 1.25, 0.03
5000, 1.31, 0.18
...
0.01, 8.90, 6.42
```

`neg_im_z_ohm` is -Im(Z) — positive in the standard Nyquist convention.

Place all CSV files directly inside the `my_data/` folder.

**Format B — Pre-fitted Rct time series (single CSV)**
If you already have Rct values extracted from your spectra, skip the fitting step.
Place a single file called `rct_series.csv` inside the `my_data/` folder:

```
measurement_index, rct_ohm
0, 8.1
1, 8.3
...
```

---

**If you use this notebook on real data, please open an issue or reach out on X [@02iTAG](https://x.com/02iTAG) — the results are useful regardless of whether the detector works or not.**

In [ ]:
# ── CONFIGURATION — edit these values ─────────────────────────────────────────

# Choose input format: 'spectra' or 'rct_series'
INPUT_FORMAT = 'spectra'

# Format A: folder containing one CSV per EIS measurement, named to sort chronologically
# Place your CSV files in the my_data/ folder next to this notebook
SPECTRA_FOLDER = 'my_data/'

# Format B: single CSV with columns [measurement_index, rct_ohm]
# Place your file in the my_data/ folder next to this notebook
RCT_CSV_PATH = 'my_data/rct_series.csv'

# If you know when the electrode failed or was replaced, enter that measurement index.
# Leave as None if unknown.
KNOWN_FAILURE_INDEX = None   # e.g. 150

# CUSUM parameters — start with these defaults, tune if needed
CUSUM_THRESHOLD = 5.0   # lower = more sensitive, more false alarms
CUSUM_DRIFT     = 0.05  # fraction of baseline Rct to ignore per sample

# Number of initial measurements to use as the healthy baseline
N_BASELINE = 10

# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from randles_model import moe_anode_baseline, moe_frequencies
from detector import fit_spectrum, cusum_detector
from randles_model import RandlesParams

print('Libraries loaded.')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────

if INPUT_FORMAT == 'spectra':
    files = sorted(glob.glob(os.path.join(SPECTRA_FOLDER, '*.csv')))
    if not files:
        raise FileNotFoundError(f'No CSV files found in {SPECTRA_FOLDER}')
    print(f'Found {len(files)} spectrum files. Fitting Randles model to each...')

    fitted_params = []
    failed = []
    prev = moe_anode_baseline()

    for i, fpath in enumerate(files):
        df = pd.read_csv(fpath)
        freqs = df['frequency_hz'].values
        Z = df['re_z_ohm'].values - 1j * df['neg_im_z_ohm'].values
        try:
            p = fit_spectrum(Z, freqs, initial_guess=prev)
            fitted_params.append(p)
            prev = p
        except Exception as e:
            print(f'  Warning: fit failed for {os.path.basename(fpath)} ({e}) — carrying forward previous fit')
            fitted_params.append(prev)
            failed.append(i)

    rct_series = np.array([p.Rct for p in fitted_params])
    rs_series  = np.array([p.Rs  for p in fitted_params])
    cdl_series = np.array([p.Cdl for p in fitted_params])
    indices    = np.arange(len(rct_series))
    print(f'Done. {len(fitted_params) - len(failed)} fits succeeded, {len(failed)} carried forward.')

elif INPUT_FORMAT == 'rct_series':
    df = pd.read_csv(RCT_CSV_PATH)
    indices    = df['measurement_index'].values
    rct_series = df['rct_ohm'].values
    rs_series  = None
    cdl_series = None
    print(f'Loaded {len(rct_series)} Rct values from {RCT_CSV_PATH}')

else:
    raise ValueError(f'INPUT_FORMAT must be "spectra" or "rct_series", got "{INPUT_FORMAT}"')

print(f'Rct range: {rct_series.min():.4f} – {rct_series.max():.4f} Ω')
print(f'Baseline Rct (first {N_BASELINE} measurements): {rct_series[:N_BASELINE].mean():.4f} Ω')

In [ ]:
# ── Run CUSUM detector ────────────────────────────────────────────────────────

baseline_rct = rct_series[:N_BASELINE].mean()
failure_hour = float(KNOWN_FAILURE_INDEX) if KNOWN_FAILURE_INDEX is not None else None

result = cusum_detector(
    rct_series, indices,
    baseline_rct=baseline_rct,
    threshold=CUSUM_THRESHOLD,
    drift=CUSUM_DRIFT,
    failure_hour=failure_hour,
)

print('=' * 50)
if result.alert_index is not None:
    print(f'ALERT fired at measurement #{result.alert_hour:.0f}')
    if failure_hour is not None:
        if result.lead_hours and result.lead_hours > 0:
            print(f'Lead time: {result.lead_hours:.0f} measurements before known failure')
        else:
            print('Alert fired AFTER known failure — try lowering CUSUM_THRESHOLD')
    else:
        print('No known failure index provided — cannot compute lead time')
else:
    print('No alert fired — electrode may still be healthy, or CUSUM_THRESHOLD is too high')
    print('Try lowering CUSUM_THRESHOLD')
print('=' * 50)

In [ ]:
# ── Plot results ──────────────────────────────────────────────────────────────

n_plots = 3 if (rs_series is not None and cdl_series is not None) else 2
fig, axes = plt.subplots(n_plots, 1, figsize=(12, 4 * n_plots), sharex=True)

# Rct trajectory
axes[0].plot(indices, rct_series, color='darkorange', lw=1.8, label='Rct')
axes[0].axhline(baseline_rct, color='grey', ls='--', lw=0.8, label=f'baseline ({baseline_rct:.3f} Ω)')
if KNOWN_FAILURE_INDEX is not None:
    axes[0].axvline(KNOWN_FAILURE_INDEX, color='red', ls='--', lw=1.5, label='known failure')
if result.alert_hour is not None:
    axes[0].axvline(result.alert_hour, color='green', ls=':', lw=1.8,
                    label=f'CUSUM alert (#{result.alert_hour:.0f})')
axes[0].set_ylabel('Rct  /  Ω')
axes[0].set_title('Charge-transfer resistance over time')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# CUSUM statistic
axes[1].plot(indices, result.statistic, color='purple', lw=1.8, label='CUSUM S[n]')
axes[1].axhline(CUSUM_THRESHOLD, color='black', ls='--', lw=1, label=f'threshold H={CUSUM_THRESHOLD}')
if KNOWN_FAILURE_INDEX is not None:
    axes[1].axvline(KNOWN_FAILURE_INDEX, color='red', ls='--', lw=1.5)
if result.alert_hour is not None:
    axes[1].axvline(result.alert_hour, color='green', ls=':', lw=1.8)
axes[1].set_ylabel('CUSUM statistic')
axes[1].set_title('CUSUM — accumulated evidence of drift')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Rs and Cdl (only if full spectra were fitted)
if n_plots == 3:
    ax3 = axes[2]
    ax3b = ax3.twinx()
    ax3.plot(indices, rs_series, color='steelblue', lw=1.5, label='Rs')
    ax3b.plot(indices, cdl_series * 1e6, color='tomato', lw=1.5, ls='--', label='Cdl')
    ax3.set_ylabel('Rs  /  Ω', color='steelblue')
    ax3b.set_ylabel('Cdl  /  μF', color='tomato')
    ax3.set_title('Supporting parameters (Rs and Cdl)')
    ax3.grid(True, alpha=0.3)
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3b.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

axes[-1].set_xlabel('Measurement index')
plt.tight_layout()
plt.savefig('../figures/05-your-own-data.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to figures/05-your-own-data.png')

In [ ]:
# ── Threshold sensitivity ─────────────────────────────────────────────────────
# Run CUSUM at multiple thresholds so you can tune for your data

print(f'Threshold sensitivity (baseline Rct = {baseline_rct:.4f} Ω):')
print(f'{"Threshold":>10}  {"Alert at":>10}  {"Lead time":>12}  {"Before failure?":>16}')
print('-' * 55)

for thr in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0]:
    r = cusum_detector(rct_series, indices, baseline_rct=baseline_rct,
                       threshold=thr, drift=CUSUM_DRIFT, failure_hour=failure_hour)
    if r.alert_hour is not None:
        lead = f'{r.lead_hours:.0f}' if r.lead_hours is not None else 'unknown'
        before = (r.alert_hour < KNOWN_FAILURE_INDEX) if KNOWN_FAILURE_INDEX else '?'
        print(f'{thr:>10.1f}  {r.alert_hour:>10.0f}  {lead:>12}  {str(before):>16}')
    else:
        print(f'{thr:>10.1f}  {"no alert":>10}  {"-":>12}  {"-":>16}')